# Census 2020 CBG 65+ Download

Downloads 2020 Census DHC age counts at the Census Block Group level and writes `census2020_cbg.csv` with only `GEOID` and `prop_65plus`.

In [ ]:
# ------------------ Libraries ------------------
library(tidycensus)
library(dplyr)
library(purrr)
library(readr)

options(tigris_use_cache = TRUE)

# ------------------ API Key ------------------
census_api_key("9e1d9e8fd17678919125e0a6ced1eab1ad651877", overwrite = TRUE, install = FALSE)

# ------------------ Output path ------------------
out_dir <- "/Users/karwailin/Desktop/CALE/data"
out_file <- file.path(out_dir, "census2020_cbg.csv")
dir.create(out_dir, showWarnings = FALSE, recursive = TRUE)

In [ ]:
# ------------------ Build DHC P12 variable map ------------------
age_65plus_groups <- c(
  "65 and 66 years",
  "67 to 69 years",
  "70 to 74 years",
  "75 to 79 years",
  "80 to 84 years",
  "85 years and over"
)

vars_dhc_2020 <- load_variables(2020, "dhc", cache = TRUE)

p12_65plus_map <- vars_dhc_2020 %>%
  filter(grepl("^P12_", name)) %>%
  filter(grepl("SEX BY AGE", concept, ignore.case = TRUE)) %>%
  filter(grepl("!!(Male|Female):", label)) %>%
  filter(Reduce(`|`, lapply(age_65plus_groups, function(g) grepl(paste0("!!", g, "$"), label)))) %>%
  transmute(variable = name)

p12_65plus_vars <- p12_65plus_map$variable
p12_65plus_vars

In [ ]:
# ------------------ Download helpers ------------------
state_list <- c(state.abb, "DC")

get_cbg_65plus_counts <- function() {
  message("Downloading DHC P12 65+ variables for Census Block Groups ...")
  map_dfr(
    state_list,
    ~ possibly(function(st) {
      get_decennial(
        geography = "block group",
        variables = p12_65plus_vars,
        year = 2020,
        sumfile = "dhc",
        state = st,
        geometry = FALSE
      ) %>%
        group_by(GEOID) %>%
        summarise(pop_65plus = sum(value, na.rm = TRUE), .groups = "drop")
    }, otherwise = NULL)(.x)
  )
}

get_cbg_total_population <- function() {
  message("Downloading PL P1 total population for Census Block Groups ...")
  map_dfr(
    state_list,
    ~ possibly(function(st) {
      get_decennial(
        geography = "block group",
        variables = "P1_001N",
        year = 2020,
        sumfile = "pl",
        state = st,
        geometry = FALSE
      ) %>%
        transmute(GEOID, total_pop = value)
    }, otherwise = NULL)(.x)
  )
}

In [ ]:
# ------------------ Run CBG download and save output ------------------
cbg_65plus <- get_cbg_65plus_counts()
cbg_totals <- get_cbg_total_population()

census2020_cbg <- cbg_65plus %>%
  left_join(cbg_totals, by = "GEOID") %>%
  transmute(
    GEOID = as.character(GEOID),
    prop_65plus = if_else(total_pop > 0, round(pop_65plus / total_pop, 6), NA_real_)
  )

write_csv(census2020_cbg, out_file)

message("Written: ", out_file)
glimpse(census2020_cbg)